<a href="https://colab.research.google.com/github/JoieLiu/PL-repo/blob/main/%E3%80%8CHW1_part1_%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4_ipynb%E3%80%8D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#日常支出速算與分攤（作業一）
- 目標：從 Sheet 讀「消費紀錄」→ 計總額/分類小計/AA 分攤 → 寫回 Sheet Summary 分頁。
- AI 點子（可選）：請模型總結本週花錢習慣與建議（例如「外食過多」）。
- Sheet 欄位：date, category, item, amount, payer

GoogleSheet: https://docs.google.com/spreadsheets/d/15QDNo4Jl47puldWS1dGBE2zFdDrB6GyfCL5PK9h6uKs/edit?usp=sharing

In [73]:
import datetime
import numpy as np
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [74]:
import pandas as pd
# read data and put it in a dataframe
# 在 google 工作表載入 gsheets
gsheets = gc.open_by_url('https://docs.google.com/spreadsheets/d/15QDNo4Jl47puldWS1dGBE2zFdDrB6GyfCL5PK9h6uKs/edit?usp=sharing')

In [75]:
# 從 gsheets 的 All-whiteboard-device 載入 sheets
sheets = gsheets.worksheet('記帳日記').get_all_values()
# 將 sheets1 資料載入 pd 的 DataFrame 進行分析
df = pd.DataFrame(sheets[1:], columns=sheets[0])
# 取得最前面的5筆資料
df.head()

,日期,時間,分類,品項,金額,付款人,地點,支付方式,備註


In [76]:
import datetime

In [89]:
import datetime
import pandas as pd

# --- 1. 使用者輸入區 ---
print("--- 📝 新增記帳紀錄 ---")
date_str = input("請輸入日期 (YYYY-MM-DD): ")
time_str = input("請輸入時間 (HH:MM): ")
item = input("請輸入品項: ")
amount = input("請輸入金額: ")
category = input("請輸入類別: ")
payment_method = input("請輸入支付方式: ")
note = input("請輸入備註 (無則按 Enter): ")

# --- 2. 準備對齊後的資料清單 ---
new_entry = [
    str(date_str), str(time_str), str(category), str(item), str(amount),
    "Joie", "未註明", str(payment_method), str(note) if note else "無", str(category)
]

# --- 3. 執行寫入 (加上簡單防重複機制) ---
confirm = input(f"\n確定要將「{item} ${amount}」存入雲端嗎？(y/n): ")

if confirm.lower() == 'y':
    try:
        # 直接追加到 Google Sheets
        worksheet.append_row(new_entry, value_input_option='USER_ENTERED')
        print(f"✅ 成功！「{item}」已存入雲端。")

        # 存完後才刷新本地 df，確保你看到的是雲端最新的狀態
        all_data = worksheet.get_all_values()
        df = pd.DataFrame(all_data[1:], columns=all_data[0])

    except Exception as e:
        print(f"❌ 存檔失敗：{e}")
else:
    print("已取消存檔，不會重複送出。")

# --- 4. 顯示最後紀錄 ---
print("\n📊 目前雲端最新紀錄：")
display(df.tail(3))

--- 📝 新增記帳紀錄 ---
請輸入日期 (YYYY-MM-DD): 2026-03-05
請輸入時間 (HH:MM): 6:40
請輸入品項: 晚餐
請輸入金額: 250
請輸入類別: 飲食
請輸入支付方式: 信用卡
請輸入備註 (無則按 Enter): 沒有

確定要將「晚餐 $250」存入雲端嗎？(y/n): y
✅ 成功！「晚餐」已存入雲端。

📊 目前雲端最新紀錄：


,日期,時間,分類,品項,金額,付款人,地點,支付方式,備註,
1,2026-03-05,12:35,飲食,午餐,210,Joie,未註明,信用卡,沒有,飲食
2,2026-03-05,3:50,化妝品,口紅,230,Joie,未註明,信用卡,沒有,化妝品
3,2026-03-05,6:40,飲食,晚餐,250,Joie,未註明,信用卡,沒有,飲食


In [84]:
date_str

'2026-03-05'

In [85]:
time_str

'12:35'

In [86]:
item

'午餐'

In [87]:
amount

'210'

In [70]:
# 創建一個新的 DataFrame 來代表你要新增的資料
new_data = pd.DataFrame([{
    '日期': date_str,
    '時間': time_str,
    '品項': item,
    '金額': amount,
    '類別': category,
    '支付方式': payment_method,
    '備註': note
}])

# 使用 concat() 將新資料合併到舊的 df 中
df = pd.concat([df, new_data], ignore_index=True)

In [71]:
df

,日期,時間,分類,品項,金額,付款人,地點,支付方式,備註,,類別
0,2026-03-05,8:35,現金,早餐,50,Joie,未註明,現金,沒有,現金,NaN
1,2026-03-05,8:35,NaN,早餐,50,NaN,NaN,現金,沒有,NaN,現金


In [72]:
# 將 DataFrame 轉換成列表的列表 (list of lists)，這是 gspread 支援的資料格式
data_to_write = df.fillna("").values.tolist()

# 第一步：取得工作表物件
worksheet = gsheets.worksheet('記帳日記')

# 第二步：將資料寫入工作表
worksheet.append_rows(values=data_to_write, value_input_option='USER_ENTERED')

print("資料已成功填補空值並寫入 Google 工作表！")

資料已成功填補空值並寫入 Google 工作表！
